**Aula 02 - Regressão linear múltipla: análise e estimação**


Prof. Elson Rodrigo de Souza Santos  
E-mail: elson.rodrigo@ufabc.edu.br ou elson129@gmail.com

Universidade Federal do ABC (UFABC)  
Bacharel em Ciências Econômicas  
Econometria I - 2026.3  
Disciplina: Econometria I - 2026.3  
Universidade Federal do ABC (UFABC)  

###
**Descrição:**
Este script constitui o material de suporte para a Aula 02 de Econometria, sobre regressão linear múltipla. Utilizamos os dados do pacote wooldridge.
#

In [ ]:
# Instação dos pacotes

!pip install wooldridge pandas numpy scipy statsmodels matplotlib jinja2


In [ ]:
# Instalação das bibliotecas:

import wooldridge as woo
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import matplotlib.pyplot as plt

In [ ]:
# ------------------------------------------------------------------------------
# 1. Carregamento da base de dados e estatísticas descritivas
# ------------------------------------------------------------------------------
# Nota: o pacote é baseado no manual de econometria básica de Wooldridge (2010),
# sendo muito útil para reproduzir os modelos apresentados no livro.
wage1 = woo.data('wage1')

# Visualização inicial da estrutura
print(wage1[['wage', 'educ', 'exper', 'tenure']].info())
print(wage1[['wage', 'educ', 'exper', 'tenure']].head())

# Seleção das variáveis de interesse e estatísticas descritivas
dados = wage1[['wage', 'educ', 'exper', 'tenure']]
descritivas = dados.describe().T
descritivas['var'] = dados.var()
print("\n--- Estatísticas Descritivas ---")
print(descritivas)

In [ ]:
# ------------------------------------------------------------------------------
# 2. Estimação dos modelos (passo a passo)
# ------------------------------------------------------------------------------
# Modelo I - Modelo simples (apenas educação)
modelo_1 = smf.ols('wage ~ educ', data=wage1).fit()
print(modelo_1.summary())

In [ ]:
# Modelo II - Adicionando experiência profissional
modelo_2 = smf.ols('wage ~ educ + exper', data=wage1).fit()
print(modelo_2.summary())

In [ ]:
# Modelo III - Modelo completo (educação, experiência e tenure)
modelo_3 = smf.ols('wage ~ educ + exper + tenure', data=wage1).fit()
print(modelo_3.summary())

In [ ]:

# Modelo IV - Modelo com viés de especificação (omitindo educação)
modelo_vies = smf.ols('wage ~ exper + tenure', data=wage1).fit()
print(modelo_vies.summary())

In [ ]:
# ------------------------------------------------------------------------------
# 3. Diagnóstico de normalidade dos resíduos (modelo_3)
# ------------------------------------------------------------------------------
residuos_3 = modelo_3.resid

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Histograma dos resíduos com curva normal teórica
axes[0].hist(residuos_3, bins=30, density=True, color='#DCE6F1', edgecolor='white', label='Resíduos')
x_grid = np.linspace(residuos_3.min(), residuos_3.max(), 200)
axes[0].plot(x_grid, stats.norm.pdf(x_grid, loc=0, scale=residuos_3.std(ddof=1)),
             color='#B22222', lw=2, linestyle='--', label='Normal Teórica')
axes[0].set_title('Distribuição dos Resíduos (modelo_3)')
axes[0].set_xlabel('Resíduos ($\\hat{u}_i$)')
axes[0].set_ylabel('Densidade')
axes[0].legend()

# Normal Q-Q Plot com pontos sólidos (equivalente a pch=19)
sm.qqplot(residuos_3, line='s', ax=axes[1], marker='o', markersize=5,
          markerfacecolor='black', markeredgecolor='black', alpha=0.5)
axes[1].set_title('Normal Q-Q Plot')

plt.tight_layout()
plt.show()

# Teste formal de normalidade de Shapiro-Wilk
stat_sw, p_val_sw = stats.shapiro(residuos_3)
print(f"\nTeste de Shapiro-Wilk: Estatística W = {stat_sw:.4f}, p-valor = {p_val_sw:.4e}")


In [ ]:
# ------------------------------------------------------------------------------
# 4. Gráficos de diagnóstico: modelo_3 vs. modelo_vies (comparação linha a linha)
# ------------------------------------------------------------------------------
def plot_diagnostico_comparativo(m_comp, m_vies):
    # Linha 1: Resíduos vs. Valores Ajustados
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, mod, titulo in zip(axes, [m_comp, m_vies], ['modelo_3 (Completo)', 'modelo_vies (Omitida)']):
        ax.scatter(mod.fittedvalues, mod.resid, color='black', alpha=0.4, s=25)
        ax.axhline(0, color='#B22222', linestyle='--', lw=1.5)
        ax.set_title(f'Resíduos vs. Ajustados - {titulo}')
        ax.set_xlabel('Valores Ajustados')
        ax.set_ylabel('Resíduos')
    plt.tight_layout()
    plt.show()

    # Linha 2: Normal Q-Q Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, mod, titulo in zip(axes, [m_comp, m_vies], ['modelo_3 (Completo)', 'modelo_vies (Omitida)']):
        sm.qqplot(mod.resid, line='s', ax=ax, marker='o', markersize=4,
                  markerfacecolor='black', markeredgecolor='black', alpha=0.4)
        ax.set_title(f'Normal Q-Q Plot - {titulo}')
    plt.tight_layout()
    plt.show()

    # Linha 3: Scale-Location (Homocedasticidade)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, mod, titulo in zip(axes, [m_comp, m_vies], ['modelo_3 (Completo)', 'modelo_vies (Omitida)']):
        resid_pad_sqrt = np.sqrt(np.abs(mod.get_influence().resid_studentized_internal))
        ax.scatter(mod.fittedvalues, resid_pad_sqrt, color='black', alpha=0.4, s=25)
        ax.set_title(f'Scale-Location - {titulo}')
        ax.set_xlabel('Valores Ajustados')
        ax.set_ylabel('Raiz(|Resíduos Padronizados|)')
    plt.tight_layout()
    plt.show()

    # Linha 4: Resíduos vs. Alavancagem (Leverage)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, mod, titulo in zip(axes, [m_comp, m_vies], ['modelo_3 (Completo)', 'modelo_vies (Omitida)']):
        inf = mod.get_influence()
        ax.scatter(inf.hat_matrix_diag, inf.resid_studentized_internal, color='black', alpha=0.4, s=25)
        ax.axhline(0, color='gray', linestyle='--', lw=1)
        ax.set_title(f'Resíduos vs. Alavancagem - {titulo}')
        ax.set_xlabel('Alavancagem (Leverage)')
        ax.set_ylabel('Resíduos Padronizados')
    plt.tight_layout()
    plt.show()

plot_diagnostico_comparativo(modelo_3, modelo_vies)

In [ ]:
# ------------------------------------------------------------------------------
# 5. Tabela comparativa com os principais resultados (e exportação LaTeX)
# ------------------------------------------------------------------------------
modelos = {
    'Modelo I': modelo_1,
    'Modelo II': modelo_2,
    'Modelo III': modelo_3,
    'Modelo IV': modelo_vies
}

variaveis = ['Intercept', 'educ', 'exper', 'tenure']
linhas_tabela = {}

for var in variaveis:
    linha = []
    for nome, mod in modelos.items():
        if var in mod.params.index:
            est = mod.params[var]
            ep = mod.bse[var]
            pv = mod.pvalues[var]
            sig = '***' if pv < 0.01 else ('**' if pv < 0.05 else ('*' if pv < 0.10 else ''))
            linha.append(f"{est:.4f}{sig} ({ep:.4f})")
        else:
            linha.append("-")
    linhas_tabela[var] = linha

# Estatísticas de ajuste
linhas_tabela['Observações (n)']  = [f"{int(mod.nobs)}" for mod in modelos.values()]
linhas_tabela['R2']               = [f"{mod.rsquared:.4f}" for mod in modelos.values()]
linhas_tabela['R2 Ajustado']      = [f"{mod.rsquared_adj:.4f}" for mod in modelos.values()]
linhas_tabela['Erro-padrão Reg.'] = [f"{np.sqrt(mod.scale):.4f}" for mod in modelos.values()]
linhas_tabela['Estatística F']    = [f"{mod.fvalue:.2f}" for mod in modelos.values()]

tabela_final = pd.DataFrame.from_dict(linhas_tabela, orient='index', columns=modelos.keys())
print("\n--- Tabela Comparativa de Resultados ---")
print(tabela_final.to_string())



In [ ]:
# Exportação direta para código LaTeX (Beamer)
print("\n--- Código LaTeX (Beamer) ---")
print(tabela_final.to_latex(escape=False))